## 1.  Check GPU availability
Before loading a large language model, it is important to verify whether a GPU is available.  
GPUs significantly accelerate inference and reduce memory constraints when working with large models.

The command `nvidia-smi` shows information about the available NVIDIA GPUs, their memory usage, and the installed drivers.

If the command is not found, it usually means that:
- the system does not have an NVIDIA GPU, or
- CUDA drivers are not installed.

In that case, the model will run on CPU, although with slower performance.

In [2]:
# ============================================================
# Check GPU availability
# ============================================================

# Displays GPU information to verify that CUDA is available
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 2. Install required libraries

We install the main libraries required to run the model:

- `transformers`: Hugging Face library for working with large language models.
- `accelerate`: helps manage hardware acceleration automatically.
- `bitsandbytes`: allows loading large models using 8-bit or 4-bit quantization to reduce memory consumption.

In [6]:
# ============================================================
# Import required libraries
# ============================================================

import torch
import torch.nn.functional as F
import pandas as pd
import time

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## 3. Authenticate with Hugging Face

To access some large language models (such as LLaMA), authentication with Hugging Face is required.

You need:

- A Hugging Face account (https://huggingface.co)
- A User Access Token, which can be generated from your account settings:
  - Go to Settings → Access Tokens
  - Create a new token with read permissions

Once you have the token, run the following cell and paste it when prompted. This will authenticate your session and allow the notebook to download and load the model.

The `login()` function allows the notebook to authenticate using your token so that restricted models can be downloaded securely.

In [5]:
# ============================================================
# Authenticate with HuggingFace
# ============================================================

# Login is required to access restricted models such as LLaMA
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 4. Load the model and tokenizer

Load the tokenizer and the language model from the Hugging Face Hub.

The tokenizer converts text into numerical representations, while the model generates predictions based on those tokens.

If GPU acceleration is available, the model will automatically be loaded onto the GPU to improve performance.

In [8]:
# ============================================================
# Load the LLaMA model and tokenizer
# ============================================================


MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

## 5. Build the prompt

In this step, we construct the prompt that will be provided to the language model.  
The prompt defines the task and gives the model the necessary context to generate a relevant response.

Carefully designing the prompt is important because large language models are highly sensitive to the way instructions and inputs are phrased.  
A clear structure typically includes the instruction, the input text, and, in some cases, formatting that helps the model understand the expected output.

In [ ]:
def build_prompt(text: str, topic: str) -> str:
    return (
        "Text:\n"
        f"{text}\n\n"
        "Question:\n"
        f"Is this text about {topic}?\n\n"
        "Answer:"
    )

import torch
import torch.nn.functional as F
import pandas as pd


## 6. Compute YES/NO probabilities

This cell defines a function to calculate the probability that the model answers "YES" or "NO" for a given prompt.

The process works as follows:

1. Tokenize the prompt and feed it into the model.
2. Extract the logits for the next token the model would generate.
3. Identify the logits corresponding to " YES" and " NO".
4. Apply the softmax function to convert logits into probabilities.

This approach allows us to perform a zero-shot classification by interpreting the model's confidence in a binary decision.

In [ ]:
def yes_no_probability(
    model,
    tokenizer,
    prompt: str,
    device: str = "cuda"
):
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # [1, seq_len, vocab_size]

    # Logits of the next token (start of the response)
    next_token_logits = logits[0, -1, :]

    # Token IDs for YES and NO
    yes_id = tokenizer(" YES", add_special_tokens=False).input_ids[0]
    no_id  = tokenizer(" NO",  add_special_tokens=False).input_ids[0]

    yes_logit = next_token_logits[yes_id]
    no_logit  = next_token_logits[no_id]

    # Softmax ONLY between YES and NO
    probs = F.softmax(torch.stack([yes_logit, no_logit]), dim=0)

    return {
        "P_yes": probs[0].item(),
        "P_no":  probs[1].item()
    }

## 7. Binary decision based on probabilities

This function determines a binary outcome ("YES" or "NO") given a dictionary of probabilities.

The process works as follows:

1. Access the probabilities of "YES" (`P_yes`) and "NO" (`P_no`) from the input dictionary.
2. Subtract the probability of "NO" from "YES" to measure the relative confidence.
3. Compare the result to an optional `margin` to control decision sensitivity.
4. Return `1` if "YES" is more likely than "NO" considering the margin, otherwise return `0`.

This method allows for a flexible binary decision that can account for uncertainty or require a minimum confidence gap.

In [ ]:
def binary_decision(prob_dict, margin: float = 0.0):
    """
    Return 1 if YES is more likely than NO (with optional margin)
    """
    return int(prob_dict["P_yes"] - prob_dict["P_no"] > margin)

## 8. Classify texts for multiple topics and export to Excel

This function evaluates a list of texts against multiple topics using a model to estimate the probability
that each text is related to each topic. The results are saved in an Excel file.

The process works as follows:

1. Iterate over each text in the input list.
2. For each text, iterate over all topics.
3. Build a prompt combining the text and the current topic.
4. Use the model and tokenizer to compute the probability that the answer is "YES" for that topic.
5. Store the "YES" probability for each topic in a row corresponding to the text.
6. Collect all rows into a DataFrame.
7. Export the DataFrame to an Excel file at the specified `output_path`.
8. Return the DataFrame for further use if needed.

This approach allows for wide-format topic classification, where each text has a probability score for every topic.

In [9]:
def classify_texts_wide_excel(
    model,
    tokenizer,
    texts: list,
    topics: list,
    output_path: str,
    device: str = "cuda"
):
    rows = []

    for text in texts:
        row = {"text": text}

        for topic in topics:
            prompt = build_prompt(text, topic)
            probs = yes_no_probability(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device
            )
            row[f"{topic}"] = probs["P_yes"]

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_excel(output_path, index=False)
    return df

## 9. Upload files from local system (Google Colab)

This cell allows the user to upload files from their local machine into the Colab environment.

The process works as follows:

1. Import the `files` module from `google.colab`.
2. Call `files.upload()`, which opens a file selection dialog.
3. The uploaded files are returned as a dictionary where the keys are filenames
   and the values are the file contents in bytes.

This is useful for providing input data to the notebook without having to download them from an external source.

In [ ]:
from google.colab import files

uploaded = files.upload()

## 10. Load and inspect Excel data

This cell reads data from an Excel file into a pandas DataFrame and displays basic information.

The process works as follows:

1. Use `pd.read_excel()` to load the sheet named 'Final' from the file
   `"AllCoders-SpreadsheetV2_1_reconciliation.xlsx"` into a DataFrame called `all`.
2. Use `print(all.head())` to show the first few rows of the DataFrame for a quick overview.
3. Use `all.columns` to list the column names and understand the structure of the data.

This allows us to quickly verify that the data has been loaded correctly and
identify the columns available for analysis.

In [ ]:
all = pd.read_excel("Main Data.xlsx", sheet_name='Final')
print(all.head())
all.columns

## 11. Classify a subset of texts across multiple topics and export to Excel

This cell demonstrates how to perform topic classification for a subset of texts  and save the results to an Excel file, while managing memory constraints.

The process works as follows:

1. Define a list of topics to classify, e.g., political, social, and conspiratorial topics.
2. Select a subset of texts from the DataFrame `all` (rows 1 to 100 in the "sequence" column) to avoid memory issues when working with large datasets.
3. Call `classify_texts_wide_excel()` with the model, tokenizer, selected texts, and topics, saving the output in a wide-format DataFrame.
4. Export the resulting DataFrame to an Excel file `"all.xlsx"` using `pd.ExcelWriter` and assign a descriptive sheet name (`"1_100"`).
5. Use `files.download()` to download the Excel file from the Colab environment.

**Note:** For memory efficiency and performance, it is recommended **not to classify all texts  and all topics at once**, especially with large datasets or  many topics. Instead, process in batches.

In [ ]:
rows = []
#topics = ["Immigration", "Health Care", "Foreign Affairs",'Elections',"Civil Rights","Culture","Trump","Climate Change","Conspiracy"]
topics = ["Conspiratorial Logic", "The Economy in General","Donald Trump","Joe Biden","Democrats","Republicans","MAGA","Jews and Antisemitism in the US","Healthcare","Reproductive Rights","Homelessness","Immigration","Climate Change","Electric Vehicles","Elections","January 6 Insurrection","Race Relations","Resistance to Social Change or Traditional Values"]
texts = all.loc[1:100, "sequence"]

df = classify_texts_wide_excel(
    model=model,
    tokenizer=tokenizer,
    texts=texts,
    topics=topics,
    output_path="llama_zero_shot_wide.xlsx"
)


with pd.ExcelWriter("all.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="1_100", index=False)
from google.colab import files
files.download("all.xlsx")